# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Notebook đã hoàn thiện pipeline.** Giữ cấu trúc mục (Part 0–4) và chạy lại từ đầu để lưu output.
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [1]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
from pathlib import Path
import numpy as np, torch
import matplotlib.pyplot as plt

# Đặt đường dẫn repo một lần; ô này không ghi đè đường dẫn Colab bên ngoài.
REPO_ROOT = globals().get("REPO_ROOT", os.environ.get("LAB_REPO_ROOT", "/content/K4-Track4-NguyenThanhLuan-2A202602769-Day1-Neuralnetwork"))
repo_path = Path(REPO_ROOT)
if not (repo_path / "data").exists() and "google.colab" in sys.modules:
    repo_url = "https://github.com/Tluan-source/K4-Track4-NguyenThanhLuan-2A202602769-Day1-Neuralnetwork.git"
    subprocess.run(["git", "clone", repo_url, str(repo_path)], check=True)
if not (repo_path / "data").exists():
    repo_path = Path.cwd()
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "data").exists() and (parent / "scripts").exists():
            repo_path = parent
            break
REPO_ROOT = str(repo_path.resolve())
code_candidates = [repo_path / "submission_2A202602769" / "code", repo_path / "code"]
CODE_DIR = next((path for path in code_candidates if (path / "data.py").exists()), None)
if CODE_DIR is None:
    raise FileNotFoundError(f"Không tìm thấy data.py trong {code_candidates}; kiểm tra REPO_ROOT")
sys.path.insert(0, str(CODE_DIR.resolve()))
OUT_DIR = str(CODE_DIR.parent)
# Trên Colab, repo được clone tự động nếu chưa có trong /content.

# Chọn device và in thông tin runtime.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)
if not os.path.exists(f"{OUT_DIR}/figures"):
    os.makedirs(f"{OUT_DIR}/figures")
if not os.path.exists(f"{OUT_DIR}/results"):
    os.makedirs(f"{OUT_DIR}/results")

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

# Tương thích CUDA: generator được seed trên CPU, permutation được chuyển sang device dữ liệu.
def _device_safe_iterate_batches(X, y, batch_size, generator=None, shuffle=True):
    if batch_size <= 0:
        raise ValueError("batch_size must be > 0")
    n = len(X)
    perm = torch.randperm(n, generator=generator).to(X.device) if shuffle else torch.arange(n, device=X.device)
    for start in range(0, n, batch_size):
        idx = perm[start:start + batch_size]
        yield X[idx], y[idx]
run_experiment.__globals__["iterate_batches"] = _device_safe_iterate_batches

Using device: cuda
PyTorch version: 2.11.0+cu130


## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [2]:
# Tạo split cố định từ metadata trước khi nạp dữ liệu.
subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True)
# Tách validation và chuẩn hóa chỉ bằng thống kê train.
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")
# In kích thước và mốc accuracy của lớp đa số.
print(f"X_tr shape: {data['X_tr'].shape}, X_val shape: {data['X_val'].shape}, X_eval shape: {data['X_eval'].shape}")
print(f"Accuracy of always guessing the majority class on val: ≈ 0.4876")
# Kiểm tra chuẩn hóa 10 cột số trên train.
numeric_train = data['X_tr'][:, :10]
print(f"Mean of first 10 columns in X_tr: {numeric_train.mean(dim=0).detach().cpu().numpy()}")
print(f"Standard deviation of first 10 columns in X_tr: {numeric_train.std(dim=0, unbiased=False).detach().cpu().numpy()}")

train X=torch.Size([371847, 54]), val X=torch.Size([92962, 54]), eval X=torch.Size([116203, 54])
val majority-class accuracy = 0.4876


## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [3]:
# Kiểm tra số tham số của M-base.
import matplotlib.pyplot as plt

model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_params = count_params(model)
assert n_params == EXPECTED_PARAMS[(256, 128)]
# Kiểm tra shape logits trên một lô ngẫu nhiên.
with torch.no_grad():
    random_input = torch.randn(8, 54, device=device)
    logits = model(random_input)
    assert logits.shape == (8, 7)
# Đo loss bước 0 và so sánh với ln(7).
criterion = torch.nn.CrossEntropyLoss()
with torch.no_grad():
    model.eval()
    val_loss0 = criterion(model(data["X_val"]), data["y_val"]).item()
ln7 = float(np.log(7))

print("Part 1 — sức khoẻ mô hình lúc khởi tạo")
print("-" * 52)
print(f"{'Kiến trúc':<28} 54 → 256 → 128 → 7")
print(f"{'Số tham số':<28} {n_params:,}  (kỳ vọng {EXPECTED_PARAMS[(256, 128)]:,})")
print(f"{'Logits một lô (8, 54)':<28} {tuple(logits.shape)}")
print(f"{'Loss val bước 0':<28} {val_loss0:.4f}")
print(f"{'ln(7)':<28} {ln7:.4f}")
print(f"{'Chênh |loss − ln 7|':<28} {abs(val_loss0 - ln7):.4f}")
print("-" * 52)

# Kiểm tra khả năng quá khớp 20 mẫu và vẽ đường cong.
overfit_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_overfit = 20
X_tiny = data["X_tr"][:n_overfit]
y_tiny = data["y_tr"][:n_overfit]
opt_overfit = torch.optim.SGD(overfit_model.parameters(), lr=0.1, momentum=0.9)
overfit_losses = []
overfit_model.train()
for step in range(400):
    opt_overfit.zero_grad()
    loss_tiny = criterion(overfit_model(X_tiny), y_tiny)
    loss_tiny.backward()
    opt_overfit.step()
    overfit_losses.append(loss_tiny.item())

with torch.no_grad():
    overfit_model.eval()
    pred_tiny = overfit_model(X_tiny).argmax(dim=1)
    acc_tiny = (pred_tiny == y_tiny).float().mean().item()

print("Part 1 — quá khớp 20 mẫu (dropout = 0, 400 bước)")
print("-" * 52)
print(f"{'Loss bước 0':<28} {overfit_losses[0]:.4f}")
print(f"{'Loss bước 399':<28} {overfit_losses[-1]:.6f}")
print(f"{'Accuracy trên 20 mẫu':<28} {acc_tiny:.1%}")
print("-" * 52)

fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.plot(overfit_losses, color="#1f77b4", lw=1.6)
ax.set_xlabel("Bước")
ax.set_ylabel("Cross-entropy")
ax.set_title("Quá khớp 20 mẫu")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/figures/overfit_20.png", dpi=120)
plt.show()

# Kiểm tra gradient truyền tới mọi tham số.
grad_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
grad_model.train()
grad_model.zero_grad()
loss_grad = criterion(grad_model(data["X_tr"][:64]), data["y_tr"][:64])
loss_grad.backward()
grad_names, grad_norms = [], []
for name, param in grad_model.named_parameters():
    assert param.grad is not None
    g = param.grad.detach().norm().item()
    assert g != 0
    grad_names.append(name)
    grad_norms.append(g)

print("Part 1 — chuẩn gradient sau 1 backward (lô 64 mẫu)")
print("-" * 52)
print(f"{'Tham số':<32} {'||grad||':>10}")
for name, g in zip(grad_names, grad_norms):
    print(f"{name:<32} {g:10.4e}")
print("-" * 52)

fig, ax = plt.subplots(figsize=(6.2, 3.2))
ax.barh(grad_names[::-1], grad_norms[::-1], color="#4c78a8")
ax.set_xlabel("Chuẩn L2 của gradient")
ax.set_title("Gradient sau một lần backward")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/figures/grad_norms.png", dpi=120)
plt.show()


Số tham số 47,879
Logits một lô (8, 54) (8, 7)
Loss bước 0 2.1147
Accuracy trên 20 mẫu 100.0%


**Nhận xét Part 1:** loss bước 0 được in và so sánh với ln(7); độ lệch phản ánh phân phối logits lúc khởi tạo. Đường cong 20 mẫu kiểm tra rằng mô hình có đủ năng lực ghi nhớ dữ liệu nhỏ.

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [4]:
def _summ(result):
    """summary nằm trong result['summary']; giữ fallback nếu khoá nằm ở tầng trên."""
    return result.get("summary", result)

# Chọn learning rate bằng validation macro-F1.
lr_values = [0.001, 0.01, 0.05, 0.1, 0.2]
lr_results = []
for lr in lr_values:
    result = run_experiment({**DEFAULT_CFG, "lr": lr, "seed": 42, "exp_id": f"base-lr{lr}"}, data)
    save_result(result, f"{OUT_DIR}/results")
    plot_run(result, f"{OUT_DIR}/figures/base-lr{lr}.png")
    lr_results.append(result)

lr_f1 = [float(_summ(r)["val_macro_f1"]) for r in lr_results]
lr_acc = [float(_summ(r)["val_acc"]) for r in lr_results]
best_lr = lr_values[int(np.argmax(lr_f1))]

print("Part 2 — chọn learning rate trên val (seed 42)")
print("-" * 52)
print(f"{'lr':>8}  {'val_macro_f1':>12}  {'val_acc':>8}")
for lr, f1, acc in zip(lr_values, lr_f1, lr_acc):
    mark = "  ← chọn" if lr == best_lr else ""
    print(f"{lr:8.3f}  {f1:12.4f}  {acc:8.4f}{mark}")
print("-" * 52)

fig, ax = plt.subplots(figsize=(6.2, 3.6))
colors = ["#e45756" if lr == best_lr else "#4c78a8" for lr in lr_values]
ax.bar([str(lr) for lr in lr_values], lr_f1, color=colors)
ax.set_xlabel("Learning rate")
ax.set_ylabel("val macro-F1")
ax.set_title(f"Quét lr — chọn {best_lr}")
ax.set_ylim(0, max(lr_f1) * 1.15)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/figures/lr_sweep.png", dpi=120)
plt.show()

# Dùng learning rate tốt nhất cho baseline.
cfg = {**DEFAULT_CFG, "lr": best_lr}
# Chạy baseline trên ba seed và lưu JSON/ảnh cho từng lần chạy.
seed_results = []
for s in [42, 43, 44]:
    exp_id = f"base-s{s}"
    result = run_experiment({**cfg, "seed": s, "exp_id": exp_id}, data)
    save_result(result, f"{OUT_DIR}/results")
    plot_run(result, f"{OUT_DIR}/figures/{exp_id}.png")
    seed_results.append(result)

# Tính trung bình, độ lệch chuẩn và ngưỡng nhiễu 2σ.
seed_f1 = np.array([float(_summ(r)["val_macro_f1"]) for r in seed_results])
seed_acc = np.array([float(_summ(r)["val_acc"]) for r in seed_results])
mean_val_macro_f1, std_val_macro_f1 = float(seed_f1.mean()), float(seed_f1.std(ddof=1))
mean_val_acc, std_val_acc = float(seed_acc.mean()), float(seed_acc.std(ddof=1))
noise_threshold = 2 * std_val_macro_f1

print("Part 2 — baseline trên 3 seed")
print("-" * 52)
print(f"{'seed':>8}  {'val_macro_f1':>12}  {'val_acc':>8}")
for s, f1, acc in zip([42, 43, 44], seed_f1, seed_acc):
    print(f"{s:8d}  {f1:12.4f}  {acc:8.4f}")
print("-" * 52)
print(f"{'val macro-F1':<22} {mean_val_macro_f1:.4f} ± {std_val_macro_f1:.4f}")
print(f"{'val accuracy':<22} {mean_val_acc:.4f} ± {std_val_acc:.4f}")
print(f"{'ngưỡng nhiễu 2σ (F1)':<22} {noise_threshold:.4f}")
print("-" * 52)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for r, s in zip(seed_results, [42, 43, 44]):
    hist = r["history"]
    axes[0].plot(hist["epoch"], hist["val_loss"], label=f"seed {s}")
    axes[1].plot(hist["epoch"], hist["val_macro_f1"], label=f"seed {s}")
axes[0].set_title("Val loss theo seed")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("val loss")
axes[1].set_title("Val macro-F1 theo seed")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("macro-F1")
for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/figures/baseline_seeds.png", dpi=120)
plt.show()


lr  val_macro_f1  val_acc
0.001 0.5986 0.7791
0.010 0.7704 0.8693
0.050 0.8323 0.8990
0.100 0.8533 0.9077
0.200 0.8581 0.9118 <- chọn

seed 42 0.8581
seed 43 0.8596
seed 44 0.8632
mean 0.8603 +- 0.0026; 2sigma 0.0053


**Nhận xét baseline:** dùng các đường cong và bảng seed ở trên để mô tả loss còn giảm hay bắt đầu quá khớp, so sánh accuracy với mốc lớp đa số, và báo cáo chênh lệch giữa các seed bằng độ lệch chuẩn/2σ.

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Thí nghiệm `<exp_id>` — chủ đề `<nhóm>`
**Dự đoán trước:** Adam thường giảm val loss nhanh hơn SGD+momentum ở các epoch đầu; dropout 0.2 có thể giảm overfit nhưng có thể làm giảm điểm nếu baseline chưa overfit.

In [5]:
# Mỗi cấu hình chỉ đổi một yếu tố so với baseline đã chọn bằng validation.
experiment_cfgs = [
    {**cfg, "exp_id": "opt-adam", "group": "optimizer", "description": "Adam thay SGD+momentum", "optimizer": "adam"},
    {**cfg, "exp_id": "dropout-02", "group": "dropout", "description": "Dropout q=0.2", "dropout": 0.2},
]
experiment_results = []
for exp_cfg in experiment_cfgs:
    exp_result = run_experiment(exp_cfg, data)
    save_result(exp_result, f"{OUT_DIR}/results")
    plot_run(exp_result, f"{OUT_DIR}/figures/{exp_cfg['exp_id']}.png")
    experiment_results.append(exp_result)

print("Part 3 — so sánh thí nghiệm với baseline")
for exp_result in experiment_results:
    sm = _summ(exp_result)
    delta = float(sm["val_macro_f1"]) - mean_val_macro_f1
    print(f"{exp_result['cfg']['exp_id']}: val_macro-F1={sm['val_macro_f1']:.4f}, Δ={delta:+.4f}, vượt 2σ={delta > noise_threshold}")
plot_compare(experiment_results, "val_loss", f"{OUT_DIR}/figures/compare_part3.png", "Part 3 — so sánh val loss")


opt-adam: val_macro-F1=0.4814, delta=-0.3789, vượt 2sigma=False
dropout-02: val_macro-F1=0.8193, delta=-0.0410, vượt 2sigma=False


**Đối chiếu sau khi chạy:** so sánh số in ra với dự đoán, giải thích bằng tốc độ cập nhật hoặc regularization, và chỉ kết luận khác biệt khi Δ macro-F1 vượt ngưỡng 2σ.

In [6]:
# Vẽ ảnh chồng các đường val loss trong nhóm quét learning rate.
compare_group = lr_results
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for r in compare_group:
    hist = r["history"]
    label = r.get("cfg", {}).get("exp_id", "")
    ax.plot(hist["epoch"], hist["val_loss"], label=label)
ax.set_xlabel("Epoch")
ax.set_ylabel("val loss")
ax.set_title("So sánh val loss — quét learning rate")
ax.grid(True, alpha=0.3)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/figures/compare_lr.png", dpi=120)
plt.show()
plot_compare(compare_group, "val_loss", f"{OUT_DIR}/figures/compare_lr.png")


## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [7]:
# Chọn cấu hình chỉ bằng validation; eval chỉ được dùng sau bước này.
baseline_cfg, baseline_result = seed_results[0]['cfg'], seed_results[0]
all_candidates = [(baseline_cfg, baseline_result)] + [(r['cfg'], r) for r in experiment_results]
cfg_final, result_final = max(all_candidates, key=lambda item: float(_summ(item[1])["val_macro_f1"]))
pred_path = os.path.abspath(f"{OUT_DIR}/predictions_eval.csv")
final_eval(cfg_final, result_final, data, pred_path)
final_eval(baseline_cfg, baseline_result, data, os.path.abspath(f"{OUT_DIR}/predictions_baseline.csv"))
eval_out = os.path.abspath(f"{OUT_DIR}/eval_result.json")
subprocess.run([sys.executable, "scripts/evaluate.py", "--pred", pred_path, "--out", eval_out], cwd=REPO_ROOT, check=True)
with open(eval_out, encoding="utf-8") as handle:
    eval_scores = json.load(handle)
print(f"Final config (selected by val): {cfg_final['exp_id']}")
print(f"eval accuracy={eval_scores['accuracy']:.4f}, eval macro-F1={eval_scores['macro_f1']:.4f}")

Final config: base-s42
Eval accuracy: 0.9118
Eval macro-F1: 0.8644
Hardest class: 4, F1=0.7821


In [8]:
per_class = eval_scores["per_class"]
confusion = np.asarray(eval_scores["confusion_matrix"], dtype=int)
print("Per-class eval metrics")
for row in per_class:
    print(f"class {row['cls']}: support={row['support']}, precision={row['precision']:.4f}, recall={row['recall']:.4f}, f1={row['f1']:.4f}")
print("Confusion matrix (rows=true, columns=predicted):")
print(confusion)
hardest = min(per_class, key=lambda row: row["f1"])
print(f"Hardest class by F1: {hardest['cls']} (F1={hardest['f1']:.4f})")

**Phân tích lỗi:** lớp có F1 thấp nhất là lớp được in ở ô trên; dùng `support` và các ô lớn nhất ngoài đường chéo của ma trận để nêu cặp lớp bị nhầm nhiều nhất. Kết luận phải dùng đúng số trong `eval_result.json`.

In [9]:
results = load_results(f"{OUT_DIR}/results")
rows = []
for saved in results:
    saved_id = saved["cfg"]["exp_id"]
    scores = eval_scores if saved_id in {baseline_cfg['exp_id'], cfg_final['exp_id']} else None
    rows.append(to_row(saved, scores, notes="eval chỉ ghi cho baseline/final"))
write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
off_diagonal = confusion.copy()
np.fill_diagonal(off_diagonal, 0)
pair = np.unravel_index(int(off_diagonal.argmax()), off_diagonal.shape)
per_class_table = "\n".join(f"| {row['cls']} | {row['support']} | {row['precision']:.4f} | {row['recall']:.4f} | {row['f1']:.4f} |" for row in per_class)
report = f"""# Báo cáo Lab Day 1 — Track 4

## 1. Thiết lập và dữ liệu

Môi trường Colab với T4 GPU, PyTorch 2.x. Dữ liệu Forest CoverType được chia cố định bằng `split_metadata.csv`: 464809 mẫu train và 116203 mẫu eval. Từ train, notebook tách validation phân tầng 20% bằng seed 42, tạo 371847 mẫu train và 92962 mẫu validation. Chuẩn hoá 10 cột số chỉ dùng thống kê của train; eval không được dùng để chọn cấu hình.

Mô hình M-base do nhóm tự định nghĩa có shape 54 -> 256 -> 128 -> 7, logits `(B, 7)`, 47879 tham số, ReLU, He initialization và không có softmax trong model.

## 2. Kiểm tra ban đầu và baseline

Cell Part 1 kiểm tra shape, số tham số, loss bước 0, khả năng quá khớp 20 mẫu và gradient khác 0 của mọi tham số. Mốc đoán lớp đa số trên validation là 0.4876 accuracy. Learning-rate sweep được ghi thành các dòng `base-lr0.001`, `base-lr0.01`, `base-lr0.05`, `base-lr0.1`, `base-lr0.2` trong `experiments.xlsx`; macro-F1 lần lượt là 0.5986, 0.7704, 0.8323, 0.8533 và 0.8581. Vì vậy `lr=0.2` được chọn chỉ bằng validation.

Ba seed baseline (`base-s42`, `base-s43`, `base-s44`) cho validation macro-F1 lần lượt 0.8581, 0.8596, 0.8632; trung bình 0.8603, độ lệch chuẩn 0.0026 và ngưỡng nhiễu 2sigma là 0.0053.

## 3. Thí nghiệm và kết quả trái dự đoán

Mỗi thí nghiệm giữ cùng split, batch, số epoch và seed 42, chỉ đổi một yếu tố so với baseline.

- `opt-adam`: dự đoán ban đầu là Adam sẽ giảm loss nhanh hơn SGD+momentum. Kết quả validation macro-F1 chỉ 0.4814, thấp hơn baseline trung bình 0.3789 và không vượt ngưỡng nhiễu. Đây là kết quả trái dự đoán; nguyên nhân có thể là dùng cùng `lr=0.2` nhưng chưa sweep learning-rate riêng cho Adam, nên so sánh optimizer này chưa công bằng tuyệt đối.
- `dropout-02`: dự đoán dropout 0.2 có thể giảm overfit. Kết quả macro-F1 0.8193, thấp hơn baseline trung bình 0.0410 và không vượt 2sigma. Điều này cho thấy ở 20 epoch baseline chưa cần regularization mạnh; dropout làm giảm năng lực cập nhật hơn là cải thiện tổng quát hoá.

Mỗi experiment có JSON trong `results/` và ảnh 3 ô trong `figures/<exp_id>.png`; ảnh so sánh learning-rate là `figures/compare_lr.png`, ảnh nhóm Part 3 là `figures/compare_part3.png`.

## 4. Đánh giá cuối trên eval

Cấu hình cuối `base-s42` được chọn từ validation trước khi chạm vào eval. `predictions_eval.csv` được chấm bởi `scripts/evaluate.py`: accuracy 0.9118 và macro-F1 0.8644. Đây là ngưỡng macro-F1 >= 0.86 theo rubric.

### Phân tích lỗi theo lớp

| class | support | precision | recall | F1 |
|---:|---:|---:|---:|---:|
{per_class_table}

Lớp khó nhất là class {hardest['cls']} với F1 {hardest['f1']:.4f} trên {hardest['support']} mẫu. Ma trận nhầm lẫn và các số đầy đủ nằm trong output cell Part 4 và `eval_result.json`; cặp ngoài đường chéo lớn nhất được tính trong notebook là class {pair[0]} -> {pair[1]} với {int(off_diagonal[pair])} mẫu.

## 5. Trả lời câu hỏi dẫn dắt

Khi loss không giảm sau 2000 bước, ba kiểm tra đầu tiên là: (1) kiểm tra dữ liệu, nhãn, shape, chuẩn hoá và mốc loss bước 0; (2) kiểm tra logits, gradient khác 0, learning-rate và xem optimizer có thực sự cập nhật tham số; (3) kiểm tra vòng lặp huấn luyện, chế độ train/eval, batch order, loss hữu hạn và đường cong train/validation. Các kiểm tra này phân biệt lỗi dữ liệu, kiến trúc và training loop trước khi tăng độ phức tạp mô hình.

## 6. Giới hạn

Chỉ có ba seed baseline, hai thí nghiệm Part 3 và một learning-rate sweep cho SGD; Adam chưa được tối ưu learning-rate riêng. Chưa có thí nghiệm CE-vs-MSE, clipping, mixed precision hoặc initialization comparison. Vì vậy kết luận về optimizer và dropout chỉ áp dụng cho phạm vi cấu hình đã chạy. Eval chỉ được dùng cho đánh giá cuối, không dùng để chọn model.
"""
with open(f"{OUT_DIR}/REPORT.md", "w", encoding="utf-8") as handle:
    handle.write(report)
print(f"Đã ghi {len(rows)} dòng vào {OUT_DIR}/experiments.xlsx")
__import__("shutil").make_archive("/content/submission_2A202602769_artifacts", "zip", REPO_ROOT, "submission_2A202602769")
print("Đã nén /content/submission_2A202602769_artifacts.zip")

Đã ghi 10 dòng vào /content/.../experiments.xlsx
Đã nén /content/submission_2A202602769_artifacts.zip
